**3.2: Lọc dữ liệu**


In [ ]:
import pandas as pd
import numpy as np
import logging
from vnstock import Quote

# ================== CẤU HÌNH ==================
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

bank_symbols = [
    'ABB', 'ACB', 'BAB', 'BID', 'BVB', 'CTG', 'EIB', 'HDB', 'KLB', 'LPB',
    'MBB', 'MSB', 'NAB', 'NVB', 'OCB', 'PGB', 'SGB', 'SHB', 'SSB', 'STB',
    'TCB', 'TPB', 'VAB', 'VBB', 'VCB', 'VIB', 'VPB'
]

START_DATE = '2015-01-01'
END_DATE   = '2023-12-31'
MAX_MISSING_RATIO = 0.40   # loại nếu thiếu >40% số ngày trong giai đoạn

# ================== HÀM HỖ TRỢ ==================
def fetch_close_series(ticker: str, start_date: str, end_date: str):
    """
    Tải lịch sử giá 1D cho ticker từ 'vci', trả về Series Close (index = Date).
    Lỗi hoặc rỗng -> trả None.
    """
    try:
        q = Quote(source='vci', symbol=ticker)
        df = q.history(start=start_date, end=end_date, interval='1D')
        if df is None or df.empty:
            logging.warning(f"{ticker}: không có dữ liệu trong giai đoạn.")
            return None
        df = df[['time', 'close']].rename(columns={'time': 'Date', 'close': 'Close'})
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.dropna(subset=['Date']).sort_values('Date')
        df = df[~df['Date'].duplicated(keep='first')]
        ser = df.set_index('Date')['Close']
        return ser
    except Exception as e:
        logging.error(f"{ticker}: lỗi tải dữ liệu - {e}")
        return None

def build_price_panel(symbols, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Ghép tất cả Series Close thành 1 DataFrame (outer join theo union ngày),
    cắt theo khung thời gian yêu cầu.
    """
    series_dict = {}
    for tk in symbols:
        logging.info(f"Tải dữ liệu: {tk}")
        ser = fetch_close_series(tk, start_date, end_date)
        if ser is not None:
            series_dict[tk] = ser

    if not series_dict:
        raise RuntimeError("Không tải được dữ liệu cho bất kỳ mã nào.")

    df = pd.concat(series_dict, axis=1)  # columns = tickers
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    s, e = pd.to_datetime(start_date), pd.to_datetime(end_date)
    df = df.loc[(df.index >= s) & (df.index <= e)]
    return df

def filter_tickers(price_data: pd.DataFrame, max_missing_ratio: float = 0.50):
    """
    Trả về 2 list: (valid, invalid) theo tỷ lệ thiếu dữ liệu trên toàn bộ khung thời gian.
    - price_data: index = Date (union ngày), columns = tickers, values = Close
    """
    total_days = price_data.shape[0]
    if total_days == 0:
        raise ValueError("price_data không có dòng nào trong giai đoạn yêu cầu.")

    valid, invalid = [], []
    for tk in price_data.columns:
        # Tỷ lệ thiếu = số ngày NA / tổng số ngày trong panel
        miss_ratio = 1 - price_data[tk].notna().sum() / total_days
        if miss_ratio > max_missing_ratio:
            invalid.append(tk)
        else:
            valid.append(tk)
    return valid, invalid

# ================== CHẠY ==================
if __name__ == "__main__":
    price_panel = build_price_panel(bank_symbols, START_DATE, END_DATE)
    valid, invalid = filter_tickers(price_panel, MAX_MISSING_RATIO)

    print("\n✅ CỔ PHIẾU ĐẠT CHUẨN:")
    print(valid)

    print("\n❌ CỔ PHIẾU CHƯA ĐẠT:")
    print(invalid)

    # (Tuỳ chọn) Lưu ra file cho tiện theo dõi:
    # pd.Series(valid).to_csv("valid_tickers.csv", index=False)
    # pd.Series(invalid).to_csv("invalid_tickers.csv", index=False)


2025-08-20 21:11:37,100 - INFO - Tải dữ liệu: ABB
2025-08-20 21:11:38,005 - INFO - Tải dữ liệu: ACB
2025-08-20 21:11:38,973 - INFO - Tải dữ liệu: BAB
2025-08-20 21:11:39,888 - INFO - Tải dữ liệu: BID
2025-08-20 21:11:40,888 - INFO - Tải dữ liệu: BVB
2025-08-20 21:11:41,722 - INFO - Tải dữ liệu: CTG
2025-08-20 21:11:42,920 - INFO - Tải dữ liệu: EIB
2025-08-20 21:11:43,855 - INFO - Tải dữ liệu: HDB
2025-08-20 21:11:44,722 - INFO - Tải dữ liệu: KLB
2025-08-20 21:11:45,522 - INFO - Tải dữ liệu: LPB
2025-08-20 21:11:46,465 - INFO - Tải dữ liệu: MBB
2025-08-20 21:11:47,654 - INFO - Tải dữ liệu: MSB
2025-08-20 21:11:48,522 - INFO - Tải dữ liệu: NAB
2025-08-20 21:11:49,291 - INFO - Tải dữ liệu: NVB
2025-08-20 21:11:50,241 - INFO - Tải dữ liệu: OCB
2025-08-20 21:11:51,022 - INFO - Tải dữ liệu: PGB
2025-08-20 21:11:51,773 - INFO - Tải dữ liệu: SGB
2025-08-20 21:11:52,572 - INFO - Tải dữ liệu: SHB
2025-08-20 21:11:53,573 - INFO - Tải dữ liệu: SSB
2025-08-20 21:11:54,389 - INFO - Tải dữ liệu: STB



✅ CỔ PHIẾU ĐẠT CHUẨN:
['ACB', 'BAB', 'BID', 'CTG', 'EIB', 'HDB', 'KLB', 'LPB', 'MBB', 'NVB', 'SHB', 'STB', 'TCB', 'TPB', 'VCB', 'VIB', 'VPB']

❌ CỔ PHIẾU CHƯA ĐẠT:
['ABB', 'BVB', 'MSB', 'NAB', 'OCB', 'PGB', 'SGB', 'SSB', 'VAB', 'VBB']


**3.3 Tối ưu hóa tỷ trọng danh mục để tối đa hóa Sharpe Ratio**

In [14]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, differential_evolution, dual_annealing
from deap import base, creator, tools, algorithms
from bayes_opt import BayesianOptimization
import random
from vnstock import Quote 

# Danh sách mã cổ phiếu
tickers = ['ACB', 'BAB', 'BID', 'CTG', 'EIB', 'HDB', 'KLB', 'LPB', 'MBB', 'NVB', 'SHB', 'STB', 'TCB', 'TPB', 'VCB', 'VIB', 'VPB']
start_date = '2015-01-01'
end_date = '2023-12-31'

# Khởi tạo DataFrame chứa dữ liệu giá đóng cửa
price_data = pd.DataFrame()

# Lấy dữ liệu lịch sử cho từng mã
for ticker in tickers:
    try:
        q = Quote(source='vci', symbol=ticker)
        df = q.history(start=start_date, end=end_date, interval='1D')

        if df is not None and not df.empty:
            df = df[['time', 'close']].copy()
            df['time'] = pd.to_datetime(df['time'])
            df.set_index('time', inplace=True)
            df.columns = [ticker]
            price_data = pd.concat([price_data, df], axis=1)
        else:
            print(f"⚠️ Không có dữ liệu cho mã: {ticker}")

    except Exception as e:
        print(f"❌ Lỗi khi xử lý {ticker}: {e}")

print("✅ Hoàn tất. Kích thước price_data:", price_data.shape)
# 🔹 **Làm sạch dữ liệu**
price_data.dropna(inplace=True)
returns = price_data.pct_change().dropna()
returns.replace([np.inf, -np.inf], np.nan, inplace=True)
returns.dropna(inplace=True)

average_returns = returns.mean()
cov_matrix = returns.cov()
risk_free_rate = 0.035 / 252  # Lãi suất phi rủi ro hàng ngày

# Kiểm tra nếu dữ liệu có lỗi
if np.isnan(returns.values).any() or np.isinf(returns.values).any():
    raise ValueError("Dữ liệu lợi nhuận có giá trị NaN hoặc vô hạn!")

# 🔹 **Hàm chuẩn hóa trọng số**
def normalize_weights(weights):
    weights = np.clip(weights, 0, 1)
    return weights / np.sum(weights)

# 🔹 **Hàm tính toán Sharpe Ratio**
def negative_sharpe_ratio(weights):
    weights_normalized = normalize_weights(weights)
    portfolio_ret = np.dot(weights_normalized, average_returns)
    portfolio_vol = np.sqrt(np.dot(weights_normalized.T, np.dot(cov_matrix, weights_normalized)))

    if np.isnan(portfolio_vol) or portfolio_vol == 0:
        return float('inf')

    sharpe = (portfolio_ret - risk_free_rate) / portfolio_vol
    return -sharpe if np.isfinite(sharpe) else float('inf')

# 🔹 **Bayesian Optimization (Tối ưu hóa Sharpe Ratio)**
def bayesian_optimization():
    def objective(**weights):
        weight_array = np.array([weights[ticker] for ticker in tickers])
        sharpe = negative_sharpe_ratio(weight_array)
        return -sharpe if np.isfinite(sharpe) else None  # Tránh lỗi

    pbounds = {ticker: (0, 1) for ticker in tickers}
    optimizer = BayesianOptimization(f=objective, pbounds=pbounds, random_state=42)

    try:
        optimizer.maximize(init_points=5, n_iter=50)
        best_weights = optimizer.max['params']
        return normalize_weights(np.array([best_weights[ticker] for ticker in tickers]))
    except Exception as e:
        print(f"Lỗi trong Bayesian Optimization: {e}")
        return None

# 🔹 **Simulated Annealing (SA)**
num_assets = len(average_returns)
bounds = [(0, 1) for _ in range(num_assets)]
result_sa = dual_annealing(negative_sharpe_ratio, bounds=bounds)
optimal_weights_sa = normalize_weights(result_sa.x)

# 🔹 **Genetic Algorithm (GA)**
creator.create("FitnessMax", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_float", random.uniform, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=num_assets)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individual):
    return (negative_sharpe_ratio(normalize_weights(individual)),)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0.5, sigma=0.2, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

population = toolbox.population(n=100)
algorithms.eaSimple(population, toolbox, cxpb=0.7, mutpb=0.2, ngen=50, verbose=False)
best_ind = tools.selBest(population, k=1)[0]
optimal_weights_ga = normalize_weights(best_ind)

# 🔹 **Phân bổ tỷ trọng đều**
equal_weights = np.array([1 / num_assets] * num_assets)

# 🔹 **Tính toán hiệu suất danh mục**
def calculate_performance(weights):
    annual_return = np.dot(weights, average_returns) * 252
    annual_risk = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
    sharpe_ratio = (annual_return - risk_free_rate) / annual_risk if annual_risk > 0 else 0
    return annual_return, annual_risk, sharpe_ratio

performance_sa = calculate_performance(optimal_weights_sa)
performance_ga = calculate_performance(optimal_weights_ga)
performance_equal = calculate_performance(equal_weights)

# 🔹 **Tối ưu hóa bằng Bayesian Optimization**
optimal_weights_bayes = bayesian_optimization()
performance_bayes = calculate_performance(optimal_weights_bayes) if optimal_weights_bayes is not None else None

# 🔹 **In kết quả**
def print_results(name, tickers, weights, performance):
    print(f"\n🔹 **Kết quả tối ưu hóa {name}:**")
    print("  📊 **Tỷ trọng tối ưu:**")
    for ticker, weight in zip(tickers, weights):
        print(f"    {ticker}: {weight * 100:.2f}%")
    print(f"  📈 **Lợi suất kỳ vọng hàng năm:** {performance[0]:.2%}")
    print(f"  📉 **Rủi ro hàng năm:** {performance[1]:.2%}")
    print(f"  📊 **Sharpe Ratio:** {performance[2]:.2f}")

print_results("Simulated Annealing", tickers, optimal_weights_sa, performance_sa)
print_results("Genetic Algorithm", tickers, optimal_weights_ga, performance_ga)
print_results("Equal Weight Portfolio", tickers, equal_weights, performance_equal)

if performance_bayes is not None:
    print_results("Bayesian Optimization", tickers, optimal_weights_bayes, performance_bayes)
else:
    print("\n❌ **Bayesian Optimization thất bại!**")


✅ Hoàn tất. Kích thước price_data: (2249, 17)


A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.


|   iter    |  target   |    ACB    |    BAB    |    BID    |    CTG    |    EIB    |    HDB    |    KLB    |    LPB    |    MBB    |    NVB    |    SHB    |    STB    |    TCB    |    TPB    |    VCB    |    VIB    |    VPB    |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | 0.03043   | 0.3745    | 0.9507    | 0.732     | 0.5987    | 0.156     | 0.156     | 0.05808   | 0.8662    | 0.6011    | 0.7081    | 0.02058   | 0.9699    | 0.8324    | 0.2123    | 0.1818    | 0.1834    | 0.3042    |
| 2         | 0.03542   | 0.5248    | 0.4319    | 0.2912    | 0.6119    | 0.1395    | 0.2921    | 0.3664    | 0.4561    | 0.7852    | 0.1997    | 0.5142    | 0.5924    | 0.04645   | 0.6075    | 0.1705    | 0.06505   | 0.9489    |
| 3         | 0.03579   | 0.9656    | 0.8084    | 0.3046    | 0.09767   | 0.6842

In [15]:
import numpy as np

# So sánh Sharpe Ratio của bốn phương pháp
sharpe_ratios = {
    "Simulated Annealing": performance_sa[2],
    "Genetic Algorithm": performance_ga[2],
    "Equal Weighting": performance_equal[2]
}

# Nếu Bayesian Optimization có kết quả hợp lệ, thêm vào danh sách so sánh
if performance_bayes is not None:
    sharpe_ratios["Bayesian Optimization"] = performance_bayes[2]

# Chọn phương pháp có Sharpe Ratio cao nhất
better_method = max(sharpe_ratios, key=sharpe_ratios.get)

# Gán trọng số và hiệu suất theo phương pháp tốt nhất
if better_method == "Simulated Annealing":
    better_weights = optimal_weights_sa
    better_performance = performance_sa
elif better_method == "Genetic Algorithm":
    better_weights = optimal_weights_ga
    better_performance = performance_ga
elif better_method == "Bayesian Optimization":
    better_weights = optimal_weights_bayes
    better_performance = performance_bayes
else:
    better_weights = equal_weights
    better_performance = performance_equal

# In kết quả so sánh
print(f"\n✅ **Phương án tối ưu nhất: {better_method}**")
print(f"  📈 **Lợi suất kỳ vọng hàng năm:** {better_performance[0]:.2%}")
print(f"  📉 **Rủi ro hàng năm:** {better_performance[1]:.2%}")
print(f"  📊 **Sharpe Ratio:** {better_performance[2]:.2f}")

# Hiển thị danh mục tối ưu
print("\n📊 **Tỷ trọng danh mục tối ưu:**")
for ticker, weight in zip(tickers, better_weights):
    if weight > 0:
        print(f"    {ticker}: {weight * 100:.2f}%")



✅ **Phương án tối ưu nhất: Simulated Annealing**
  📈 **Lợi suất kỳ vọng hàng năm:** 21.36%
  📉 **Rủi ro hàng năm:** 23.85%
  📊 **Sharpe Ratio:** 0.90

📊 **Tỷ trọng danh mục tối ưu:**
    BID: 0.37%
    EIB: 13.25%
    KLB: 9.42%
    LPB: 19.93%
    NVB: 9.03%
    SHB: 14.38%
    STB: 0.37%
    TPB: 3.04%
    VCB: 30.22%


**3.4: Tối ưu hóa các thông số "Short" và "Long" của MA**


In [16]:
import numpy as np
import pandas as pd
from backtesting import Backtest, Strategy
from ta.trend import sma_indicator, ema_indicator, wma_indicator
from vnstock import Quote
from joblib import Parallel, delayed
import logging

# =========================
# Cấu hình logging
# =========================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

selected_tickers = [ticker for ticker, weight in zip(tickers, better_weights) if weight > 0]

# =========================
# Hàm lấy dữ liệu chứng khoán
# =========================
def get_stock_data(ticker):
    try:
        q = Quote(source='vci', symbol=ticker)
        data = q.history(start='2015-01-01', end='2023-12-31', interval='1D')

        if data is None or data.empty:
            logging.warning(f"⚠️ Không có dữ liệu cho mã {ticker}")
            return None

        data.rename(columns={'open': 'Open', 'high': 'High', 'low': 'Low', 
                             'close': 'Close', 'volume': 'Volume'}, inplace=True)
        data['time'] = pd.to_datetime(data['time'])
        data.set_index('time', inplace=True)
        data = data[['Open', 'High', 'Low', 'Close', 'Volume']]
        data.fillna(method='ffill', inplace=True)

        return data

    except Exception as e:
        logging.error(f"❌ Lỗi khi lấy dữ liệu mã {ticker}: {e}")
        return None

# =========================
# Chiến lược giao dịch MA
# =========================
class MAStrategy(Strategy):
    ma_type = 'SMA'
    short_period = 10
    long_period = 50

    def init(self):
        ma_func = {'SMA': sma_indicator, 'EMA': ema_indicator, 'WMA': wma_indicator}[self.ma_type]
        self.ma_short = self.I(lambda x: ma_func(pd.Series(x), window=self.short_period), self.data.Close)
        self.ma_long = self.I(lambda x: ma_func(pd.Series(x), window=self.long_period), self.data.Close)

    def next(self):
        if self.ma_short[-1] > self.ma_long[-1] and self.ma_short[-2] <= self.ma_long[-2]:
            self.buy()
        elif self.ma_short[-1] < self.ma_long[-1] and self.ma_short[-2] >= self.ma_long[-2]:
            self.position.close()

# =========================
# Hàm backtest để tính Sharpe Ratio
# =========================
def backtest_ma(short, long, ma_type, data):
    if short >= long:
        return -1  
    
    ma_func = {'SMA': sma_indicator, 'EMA': ema_indicator, 'WMA': wma_indicator}[ma_type]
    data[f'{ma_type} Short'] = ma_func(pd.Series(data['Close']), window=int(short))
    data[f'{ma_type} Long'] = ma_func(pd.Series(data['Close']), window=int(long))

    bt = Backtest(data, MAStrategy, cash=100000, commission=0.002)
    stats = bt.run(short_period=int(short), long_period=int(long), ma_type=ma_type)
    return stats['Sharpe Ratio']

# =========================
# Grid Search Optimization (lấy top 10%)
# =========================
def optimize_gridsearch(ticker, ma_type, top_percent=0.1):
    data = get_stock_data(ticker)
    if data is None:
        return None
    
    results = []
    for short in range(5, 50, 5):
        for long in range(51, 200, 10):
            sharpe = backtest_ma(short, long, ma_type, data)
            results.append({'Ticker': ticker, 'MA': ma_type,
                            'Short': short, 'Long': long, 'Sharpe Ratio': sharpe})
    
    df = pd.DataFrame(results)
    df = df[df['Sharpe Ratio'] > -1]  # loại bỏ cặp invalid
    
    if df.empty:
        return None
    
    # Sắp xếp theo Sharpe Ratio giảm dần
    df_sorted = df.sort_values(by='Sharpe Ratio', ascending=False).reset_index(drop=True)
    
    # Lấy top 10%
    top_n = max(1, int(len(df_sorted) * top_percent))  
    df_top = df_sorted.head(top_n)
    
    return df_top

# =========================
# Chạy tối ưu hóa song song
# =========================
results = {
    'GridSearch': {
        ma: Parallel(n_jobs=-1)(
                delayed(optimize_gridsearch)(ticker, ma, top_percent=0.1) 
                for ticker in selected_tickers
            ) 
        for ma in ['SMA', 'EMA', 'WMA']
    }
}

# =========================
# Tạo DataFrame kết quả
# =========================
final_results = {
    'GridSearch': {
        ma: pd.concat([res for res in result if res is not None], ignore_index=True)
        for ma, result in results['GridSearch'].items()
    }
}

# =========================
# Hiển thị kết quả
# =========================
for ma, df in final_results['GridSearch'].items():
    print(f"\n📈 Top 10% {ma} Parameters using GridSearch:")
    print(df)




📈 Top 10% SMA Parameters using GridSearch:
    Ticker   MA  Short  Long  Sharpe Ratio
0      BID  SMA      5    51      0.520603
1      BID  SMA      5    61      0.457696
2      BID  SMA     10    51      0.397147
3      BID  SMA     30    51      0.360368
4      BID  SMA     20    51      0.346019
5      BID  SMA     15    51      0.313353
6      BID  SMA     10    71      0.290208
7      BID  SMA      5    71      0.273395
8      BID  SMA     40   171      0.259273
9      BID  SMA     25    51      0.258613
10     BID  SMA     10    81      0.258145
11     BID  SMA     10    61      0.254362
12     BID  SMA     45   171      0.228866
13     EIB  SMA      5    91      0.152116
14     EIB  SMA      5   191      0.127706
15     EIB  SMA      5   101      0.114922
16     EIB  SMA      5   111      0.059715
17     EIB  SMA      5    81      0.050953
18     EIB  SMA      5   131      0.044968
19     EIB  SMA      5   121      0.042913
20     EIB  SMA     25   111      0.029674
21     EIB

In [17]:
grid_results = {}

for ma, df in final_results['GridSearch'].items():
    df = df.copy()
    df["Weight"] = df["Ticker"].map(dict(zip(tickers, better_weights)))
    df = df[df["Weight"] > 0]   # chỉ giữ ticker có trọng số > 0
    grid_results[ma] = df
    print(f"\n📈 Top 10% {ma} Parameters using GridSearch:")
    
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    pd.set_option("display.max_colwidth", None)
    print(df)


📈 Top 10% SMA Parameters using GridSearch:
    Ticker   MA  Short  Long  Sharpe Ratio    Weight
0      BID  SMA      5    51      0.520603  0.003677
1      BID  SMA      5    61      0.457696  0.003677
2      BID  SMA     10    51      0.397147  0.003677
3      BID  SMA     30    51      0.360368  0.003677
4      BID  SMA     20    51      0.346019  0.003677
5      BID  SMA     15    51      0.313353  0.003677
6      BID  SMA     10    71      0.290208  0.003677
7      BID  SMA      5    71      0.273395  0.003677
8      BID  SMA     40   171      0.259273  0.003677
9      BID  SMA     25    51      0.258613  0.003677
10     BID  SMA     10    81      0.258145  0.003677
11     BID  SMA     10    61      0.254362  0.003677
12     BID  SMA     45   171      0.228866  0.003677
13     EIB  SMA      5    91      0.152116  0.132517
14     EIB  SMA      5   191      0.127706  0.132517
15     EIB  SMA      5   101      0.114922  0.132517
16     EIB  SMA      5   111      0.059715  0.132517
17

**3.5 Phương pháp kiểm định chiến lược giao dịch**


In [18]:
# =========================
# ENSEMBLE TOP-K (SMA/EMA/WMA) — TRAIN 2015-2023, TEST 2024-2025
# PHƯƠNG ÁN 2: ĐÁNH GIÁ RIÊNG TỪNG LOẠI MA CHO TOÀN DANH MỤC + RÀNG BUỘC VN
# =========================
import pandas as pd
import numpy as np
from vnstock import Quote
from ta.trend import sma_indicator, ema_indicator, wma_indicator

# ==== THAM SỐ CHUNG ====
K = 3                         # số ứng viên top để ensemble cho MỖI MÃ
VOTE_THRESHOLD = 0.5          # >50% phiếu "mua" thì vào lệnh
TCOST_BPS = 20                # phí giao dịch mỗi lần đổi trạng thái = 0.20%
RF_ANNUAL = 0.035             # lãi suất phi rủi ro năm (dùng cho Sharpe)
PERIODS = 252                 # số phiên/năm

# RÀNG BUỘC GIAO DỊCH VN
MIN_HOLD_DAYS = 2             # T+2: giữ >= 2 phiên mới được bán
STOP_LOSS_PCT = 0.07          # Stop-loss 7%
COOLDOWN_DAYS = 5             # sau SL "cấm mua lại" số phiên
T2_CASH_DAYS = 2              # tiền bán về sau 2 phiên mới tái dùng

# MỐC THỜI GIAN (Train & Test)
train_start, train_end = '2015-01-01', '2023-12-31'
test_start,  test_end  = '2024-01-01', '2025-12-31'

# =========================
# 1) TIỆN ÍCH: RÚT GỌN DANH SÁCH MÃ & TẢI GIÁ
# =========================
def needed_tickers_from_grid(grid_results: dict) -> list:
    need = []
    for _, df in grid_results.items():
        if df is None or df.empty:
            continue
        g = df[df["Weight"] > 0]
        if not g.empty:
            need.append(g["Ticker"])
    if not need:
        return []
    return sorted(set(pd.concat(need).dropna().unique().tolist()))

def fetch_prices(tickers: list, start_date: str, end_date: str) -> pd.DataFrame:
    price = pd.DataFrame()
    for t in tickers:
        try:
            q = Quote(source='vci', symbol=t)
            df = q.history(start=start_date, end=end_date, interval='1D')
            if df is not None and not df.empty:
                df = df[["time", "close"]].copy()
                df["time"] = pd.to_datetime(df["time"])
                df.set_index("time", inplace=True)
                price = pd.concat([price, df.rename(columns={"close": t})], axis=1)
            else:
                print(f"⚠️ Không có dữ liệu: {t}")
        except Exception as e:
            print(f"❌ Lỗi {t}: {e}")
    return price.sort_index()

# =========================
# 2) CHUẨN HÓA ỨNG VIÊN, MA & TÍN HIỆU
# =========================
def prepare_candidates(grid_results: dict) -> pd.DataFrame:
    req_cols = {"Ticker","MA","Short","Long","Sharpe Ratio","Weight"}
    cands_list = []
    for ma_name, df in grid_results.items():
        g = df.copy()
        if "MA" not in g.columns:
            g["MA"] = ma_name
        missing = req_cols - set(g.columns)
        if missing:
            raise ValueError(f"Thiếu cột {missing} trong grid_results['{ma_name}']")
        cands_list.append(g[list(req_cols)])
    cands = (pd.concat(cands_list, ignore_index=True)
                .dropna(subset=list(req_cols)))
    cands = cands[cands["Weight"] > 0].drop_duplicates(subset=["Ticker","MA","Short","Long"])
    cands = cands.sort_values(["Ticker","Sharpe Ratio"], ascending=[True, False]).reset_index(drop=True)
    return cands

def build_ma(series: pd.Series, ma_type: str, window: int) -> pd.Series:
    ma_type = str(ma_type).upper()
    w = int(window)
    if ma_type == "SMA":
        return sma_indicator(series, window=w, fillna=False)
    if ma_type == "EMA":
        return ema_indicator(series, window=w, fillna=False)
    if ma_type == "WMA":
        return wma_indicator(series, window=w, fillna=False)
    raise ValueError(f"Unknown MA type: {ma_type}")

def gen_signal(df_close: pd.DataFrame, ma_type: str, short_w: int, long_w: int) -> pd.Series:
    s = build_ma(df_close["close"], ma_type, int(short_w))
    l = build_ma(df_close["close"], ma_type, int(long_w))
    sig = (s > l).astype(int)  # 1=long, 0=flat
    return sig.shift(1).fillna(0)  # tránh lookahead

# =========================
# 3) METRICS
# =========================
def sharpe_annual(daily_ret: pd.Series, rf_annual: float = 0.0, periods: int = 252) -> float:
    if daily_ret.std() == 0 or len(daily_ret) == 0:
        return np.nan
    er = daily_ret.mean()*periods - rf_annual
    vol = daily_ret.std()*np.sqrt(periods)
    return er/vol if vol != 0 else np.nan

def cagr_from_nav(nav: pd.Series, periods: int = 252) -> float:
    if len(nav) < 2 or nav.iloc[0] <= 0:
        return np.nan
    total_return = nav.iloc[-1] / nav.iloc[0]
    years = len(nav) / periods
    return total_return**(1/years) - 1 if years > 0 else np.nan

def max_drawdown(nav: pd.Series) -> float:
    if len(nav) == 0:
        return np.nan
    roll_max = nav.cummax()
    drawdown = nav/roll_max - 1.0
    return drawdown.min()

def slice_series(s: pd.Series, start: str, end: str) -> pd.Series:
    if s.empty:
        return s
    mask = (s.index >= pd.to_datetime(start)) & (s.index <= pd.to_datetime(end))
    return s.loc[mask]

# =========================
# 4) ENSEMBLE & BACKTEST (áp ràng buộc VN)
# =========================
def run_ensemble_portfolio(grid_results: dict, price_data: pd.DataFrame,
                           K: int = 3, vote_threshold: float = 0.5,
                           tcost_bps: float = 2.0,
                           rf_annual: float = 0.035, periods: int = 252,
                           # VN rules:
                           min_hold_days: int = MIN_HOLD_DAYS,
                           stop_loss_pct: float = STOP_LOSS_PCT,
                           cooldown_days: int = COOLDOWN_DAYS,
                           t2_cash_days: int = T2_CASH_DAYS):
    # --- Chuẩn hóa & lấy top-K theo mã ---
    cands = prepare_candidates(grid_results)
    if cands.empty:
        raise RuntimeError("Không có ứng viên nào sau khi chuẩn hóa (candidates trống).")
    topk_by_ticker = cands.groupby("Ticker").head(K).reset_index(drop=True)

    base_signals = {}
    weights_map = {}
    picked_rows = []

    # --- Tín hiệu ensemble cơ bản ---
    for tkr, g in topk_by_ticker.groupby("Ticker"):
        if tkr not in price_data.columns:
            continue
        dfp = price_data[[tkr]].dropna().rename(columns={tkr: "close"}).sort_index()
        if dfp.empty:
            continue

        sigs = []
        for _, r in g.iterrows():
            sig = gen_signal(dfp, r["MA"], r["Short"], r["Long"])
            sigs.append(sig)
            picked_rows.append([tkr, r["MA"], int(r["Short"]), int(r["Long"]),
                                float(r["Sharpe Ratio"]), float(r["Weight"])])
        if not sigs:
            continue

        sigs_mat = pd.concat(sigs, axis=1)
        sig_ens = (sigs_mat.mean(axis=1) > vote_threshold).astype(int)
        base_signals[tkr] = sig_ens.reindex(dfp.index).fillna(0).astype(int)
        weights_map[tkr] = float(g["Weight"].iloc[0])

    if not base_signals:
        raise RuntimeError("Không có mã hợp lệ để ghép danh mục.")

    # --- Áp quy tắc VN cho từng mã ---
    def apply_vn_rules_one(sig_raw: pd.Series, px: pd.Series,
                           min_hold_days: int, stop_loss_pct: float, cooldown_days: int) -> pd.Series:
        sig = sig_raw.copy().astype(int)
        sig_adj = sig.copy()
        in_pos = False
        hold_cnt = 0
        cool_cnt = 0
        entry_price = np.nan

        for i, dt in enumerate(sig.index):
            # đang cooldown => buộc flat
            if cool_cnt > 0:
                sig_adj.iloc[i] = 0
                cool_cnt -= 1
            else:
                sig_adj.iloc[i] = 1 if sig.iloc[i] == 1 else 0

            if not in_pos and sig_adj.iloc[i] == 1:
                in_pos = True
                hold_cnt = 0
                entry_price = px.iloc[i]

            elif in_pos and sig_adj.iloc[i] == 1:
                hold_cnt += 1
                if entry_price and entry_price > 0:
                    dd = px.iloc[i] / entry_price - 1.0
                    if dd <= -abs(stop_loss_pct):
                        # thoát ngay & bật cooldown
                        sig_adj.iloc[i] = 0
                        in_pos = False
                        hold_cnt = 0
                        cool_cnt = cooldown_days
                        entry_price = np.nan

            elif in_pos and sig_adj.iloc[i] == 0:
                # muốn thoát: phải đủ T+2 bán sau mua
                if hold_cnt < (min_hold_days - 1):
                    sig_adj.iloc[i] = 1
                    hold_cnt += 1
                else:
                    in_pos = False
                    hold_cnt = 0
                    entry_price = np.nan

            if not in_pos and sig_adj.iloc[i] == 0:
                hold_cnt = 0

        return sig_adj.astype(int)

    adj_signals = {}
    for tkr, sig in base_signals.items():
        px = price_data[[tkr]].dropna().rename(columns={tkr: "close"}).sort_index()["close"]
        sig = sig.reindex(px.index).fillna(0).astype(int)
        adj_signals[tkr] = apply_vn_rules_one(
            sig_raw=sig, px=px,
            min_hold_days=min_hold_days,
            stop_loss_pct=stop_loss_pct,
            cooldown_days=cooldown_days
        )

    # --- Return & Cost từng mã ---
    per_stock_df = {}
    for tkr, sig in adj_signals.items():
        px = price_data[[tkr]].dropna().rename(columns={tkr: "close"}).sort_index()
        sig = sig.reindex(px.index).fillna(0).astype(int)

        ret = px["close"].pct_change().fillna(0.0)
        turn = sig.diff().abs().fillna(0.0)
        cost = turn * (tcost_bps / 10000.0)

        per_stock_df[tkr] = pd.DataFrame({
            "Signal": sig,
            "Return": ret,
            "Turn": turn,
            "Cost": cost,
        }, index=px.index)

    # --- T+2 tiền: scale hiệu lực trọng số theo "tiền chờ về" ---
    idx_union = sorted(set().union(*[df.index for df in per_stock_df.values()]))
    weights_df = pd.DataFrame(0.0, index=idx_union, columns=per_stock_df.keys())
    signals_mat = pd.DataFrame(0, index=idx_union, columns=per_stock_df.keys())
    returns_mat = pd.DataFrame(0.0, index=idx_union, columns=per_stock_df.keys())
    costs_mat = pd.DataFrame(0.0, index=idx_union, columns=per_stock_df.keys())

    for tkr, df in per_stock_df.items():
        w = weights_map[tkr]
        df = df.reindex(idx_union).fillna({"Signal":0,"Return":0.0,"Turn":0.0,"Cost":0.0})
        signals_mat[tkr] = df["Signal"].astype(int)
        returns_mat[tkr] = df["Return"].astype(float)
        costs_mat[tkr]   = df["Cost"].astype(float)
        weights_df[tkr]  = w

    # sự kiện SELL: 1->0
    sell_events = ((signals_mat.diff() == -1).astype(int))

    blocked_weight = pd.Series(0.0, index=idx_union)
    kernel = np.ones(t2_cash_days, dtype=float)  # ví dụ [1,1] cho 2 phiên
    for tkr in sell_events.columns:
        w_i = float(weights_df[tkr].iloc[0]) if len(weights_df[tkr]) else 0.0
        seq = (sell_events[tkr].fillna(0).astype(float) * w_i).to_numpy()
        conv = np.convolve(seq, kernel, mode='full')[:len(seq)]
        blocked_weight = blocked_weight.add(pd.Series(conv, index=idx_union), fill_value=0.0)

    blocked_weight = blocked_weight.clip(lower=0.0, upper=1.0)

    active_weight = (weights_df * signals_mat).sum(axis=1)
    capacity = (1.0 - blocked_weight).clip(lower=0.0)
    scale = pd.Series(1.0, index=idx_union)
    mask = active_weight > 1e-12
    scale.loc[mask] = np.minimum(1.0, (capacity.loc[mask] / active_weight.loc[mask]).astype(float))

    eff_weights = (weights_df * signals_mat).mul(scale, axis=0)

    # --- PnL & NAV ---
    per_stock_pnl = eff_weights * (signals_mat * returns_mat - costs_mat)
    port_ret = per_stock_pnl.sum(axis=1).rename("port_ret")
    nav = (1 + port_ret).cumprod().rename("NAV").to_frame()

    # --- Metrics ---
    m_all = {
        "Sharpe": sharpe_annual(port_ret, rf_annual=rf_annual, periods=periods),
        "CAGR":   cagr_from_nav(nav["NAV"], periods=periods),
        "MaxDD":  max_drawdown(nav["NAV"])
    }

    oos = {}
    for tag, (s, e) in {"Train": (train_start, train_end), "Test": (test_start, test_end)}.items():
        pr = slice_series(port_ret, s, e)
        if pr.empty:
            oos[tag] = {"Sharpe": np.nan, "CAGR": np.nan, "MaxDD": np.nan}
            continue
        nv = (1 + pr).cumprod()
        oos[tag] = {
            "Sharpe": sharpe_annual(pr, rf_annual=rf_annual, periods=periods),
            "CAGR":   cagr_from_nav(nv, periods=periods),
            "MaxDD":  max_drawdown(nv)
        }

    picked_df = pd.DataFrame(picked_rows, columns=["Ticker","MA","Short","Long","Sharpe Ratio","Weight"])

    return {
        "candidates": cands,
        "topk_by_ticker": topk_by_ticker,
        "picked_df": picked_df,
        "signals": {k: signals_mat[k].astype(int) for k in signals_mat.columns},  # sau VN rules
        "eff_weights": eff_weights,    # trọng số hiệu lực sau T+2 tiền
        "port_ret": port_ret,
        "nav": nav,
        "metrics_all": m_all,
        "metrics_oos": oos
    }

# =========================
# 5) XUẤT CHI TIẾT: Signal_i(t), WeightEff_i(t), Return_i(t), Cost(t), PnL_i(t)
# =========================
def export_detailed_signals(res, price_data, tcost_bps=TCOST_BPS):
    eff_w = res["eff_weights"]          # DataFrame (time × ticker)
    detailed_list = []
    for tkr in eff_w.columns:
        if tkr not in price_data.columns or tkr not in res["signals"]:
            continue
        px = price_data[[tkr]].dropna().rename(columns={tkr:"close"}).sort_index()
        idx = eff_w.index.union(px.index).sort_values()

        sig = pd.Series(res["signals"][tkr]).reindex(idx).fillna(0).astype(int)
        w   = eff_w[tkr].reindex(idx).fillna(0.0)
        ret = px["close"].reindex(idx).pct_change().fillna(0.0)
        turn= sig.diff().abs().fillna(0.0)
        cost= turn * (tcost_bps/10000.0)
        pnl = w * (sig*ret - cost)

        df_out = pd.DataFrame({
            "Ticker": tkr,
            "Signal": sig,
            "WeightEff": w,
            "Return": ret,
            "Cost": cost,
            "PnL": pnl
        }, index=idx)
        detailed_list.append(df_out)

    detailed = pd.concat(detailed_list).sort_index()
    port_pnl = detailed.groupby(level=0)["PnL"].sum().rename("Port_PnL")
    port_ret = res["port_ret"].copy().rename("port_ret")
    reconcile = pd.concat([port_pnl, port_ret], axis=1)
    reconcile["Diff"] = reconcile["Port_PnL"] - reconcile["port_ret"]
    return detailed, reconcile

# =========================
# 6) LỌC THEO PHƯƠNG PHÁP & CHẠY RIÊNG TỪNG MA
# =========================
def subset_grid_by_methods(grid_results: dict, allowed_methods=('SMA','EMA','WMA')) -> dict:
    allowed_set = {m.upper() for m in allowed_methods}
    sub = {}
    for ma_name, df in grid_results.items():
        if str(ma_name).upper() in allowed_set:
            sub[ma_name] = df.copy()
    return sub

def run_full_for_methods(grid_results_full: dict, methods=('SMA','EMA','WMA'), verbose=False):
    # Hợp nhất tickers cần cho tất cả phương pháp
    union_tickers = set()
    for m in methods:
        sub = subset_grid_by_methods(grid_results_full, allowed_methods=(m,))
        union_tickers |= set(needed_tickers_from_grid(sub))
    union_tickers = sorted(union_tickers)
    if not union_tickers:
        raise RuntimeError("Không tìm thấy ticker Weight>0 trong bất kỳ phương pháp nào.")

    # Tải giá chung
    price_data = fetch_prices(union_tickers, train_start, test_end)
    if verbose:
        print("✅ price_data (union):", price_data.shape, "| Range:", price_data.index.min(), "→", price_data.index.max())

    results = {}
    for m in methods:
        grid_sub = subset_grid_by_methods(grid_results_full, allowed_methods=(m,))
        if not grid_sub:
            results[m] = None
            continue

        res_m = run_ensemble_portfolio(
            grid_results=grid_sub,
            price_data=price_data,
            K=K,
            vote_threshold=VOTE_THRESHOLD,
            tcost_bps=TCOST_BPS,
            rf_annual=RF_ANNUAL,
            periods=PERIODS,
            min_hold_days=MIN_HOLD_DAYS,
            stop_loss_pct=STOP_LOSS_PCT,
            cooldown_days=COOLDOWN_DAYS,
            t2_cash_days=T2_CASH_DAYS
        )
        detailed_m, reconcile_m = export_detailed_signals(res_m, price_data, tcost_bps=TCOST_BPS)
        results[m] = {"res": res_m, "detailed": detailed_m, "reconcile": reconcile_m}

    return results, price_data

def print_method_summary(results_by_method: dict):
    for m, packs in results_by_method.items():
        print(f"\n===== {m} ONLY =====")
        if packs is None:
            print("No result.")
            continue
        r = packs["res"]
        allm = r["metrics_all"]
        tr = r["metrics_oos"]["Train"]; te = r["metrics_oos"]["Test"]
        print(f"ALL:   Sharpe {allm['Sharpe']:.3f} | CAGR {allm['CAGR']:.2%} | MaxDD {allm['MaxDD']:.2%}")
        print(f"Train: Sharpe {tr['Sharpe']:.3f} | CAGR {tr['CAGR']:.2%} | MaxDD {tr['MaxDD']:.2%}")
        print(f"Test:  Sharpe {te['Sharpe']:.3f} | CAGR {te['CAGR']:.2%} | MaxDD {te['MaxDD']:.2%}")

# =========================
# 7) CHẠY & XUẤT CHI TIẾT
# =========================
def run_and_export(grid_results, methods=('SMA','EMA','WMA'), out_excel="by_method_outputs_VNrules.xlsx"):
    results_by_method, price_data = run_full_for_methods(grid_results, methods=methods, verbose=False)

    # In tóm tắt 1 lần
    print_method_summary(results_by_method)

    # Xuất chi tiết
    with pd.ExcelWriter(out_excel) as w:
        for m, packs in results_by_method.items():
            if packs is None: 
                continue
            r = packs["res"]
            detailed = packs["detailed"]
            reconcile = packs["reconcile"]

            detailed.to_excel(w, sheet_name=f"{m}_Detailed", index=True)
            reconcile.to_excel(w, sheet_name=f"{m}_Reconcile", index=True)
            r["topk_by_ticker"].to_excel(w, sheet_name=f"{m}_TopK", index=False)
            r["nav"].to_excel(w, sheet_name=f"{m}_NAV")
            r["port_ret"].to_frame().to_excel(w, sheet_name=f"{m}_PortRet")

    print(f"\n✅ Đã xuất file {out_excel}")
    return results_by_method, price_data

# --- GỌI CHẠY ---
results_by_method, price_data = run_and_export(grid_results, methods=('SMA','EMA','WMA'))


The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
The default fill_method='pad' in Series.pct_change is deprecated and will be removed in 


===== SMA ONLY =====
ALL:   Sharpe 1.377 | CAGR 24.03% | MaxDD -16.21%
Train: Sharpe 1.216 | CAGR 21.34% | MaxDD -16.21%
Test:  Sharpe 2.278 | CAGR 39.70% | MaxDD -10.52%

===== EMA ONLY =====
ALL:   Sharpe 1.189 | CAGR 21.23% | MaxDD -17.81%
Train: Sharpe 1.015 | CAGR 18.35% | MaxDD -17.81%
Test:  Sharpe 2.175 | CAGR 38.13% | MaxDD -11.52%

===== WMA ONLY =====
ALL:   Sharpe 1.398 | CAGR 23.56% | MaxDD -14.89%
Train: Sharpe 1.210 | CAGR 20.48% | MaxDD -14.89%
Test:  Sharpe 2.424 | CAGR 41.72% | MaxDD -9.43%

✅ Đã xuất file by_method_outputs_VNrules.xlsx
